# Keccak Toy Implementation

<style>
    .python-function {
        color: #228B22;
        font-family: monospace;
    }
</style>

<div class="padding-10px">
    <h3>
        This <span class="python-function">keccak()</span> function is intended for educational or testing purposes only.</br> Not recommended for production use.
    </h3>
</div>

### 1️⃣ Introduction

This notebook explains Keccak in a simplified, educational manner using both mathematical notation and Python code. </br> The goal is to illustrate the sponge construction, absorption, permutation rounds, and squeezing in a toy 8-bit version.

### 2️⃣ Parameters and State

- $b$ = state size in bits  
- $r$ = rate  
- $c$ = capacity, so that $b = r + c$  
- $l$ = output length  

Permutation:  
$$
f: \{0,1\}^b \to \{0,1\}^b
$$

Message:  
$$
M \in \{0,1\}^*
$$

In [4]:
B: int = 8
R: int = 4
C: int = B - R
L: int = 4
N_ROUNDS: int = 2

### 3️⃣ Toy Keccak Permutation (f)

The permutation $f$ applies three steps per round, in order:

---

#### $\rho$ — Rotation

Shifts the state cyclically by 1 position (left rotation):

$$
\rho(S)_i = S_{(i+1) \bmod b}
$$

**Purpose**: diffuses bit positions across the state over successive rounds.  
**In real Keccak**: each of the 25 lanes is rotated by a different, fixed offset from a 5×5 table — here we simplify to a uniform shift of 1.  
**Quantum mapping**: implementable with SWAP gates (or simply a re-wiring of qubit labels — zero cost on hardware).

---

#### $\chi$ — Non-linear layer

The **only non-linear** step. Applies a bitwise function:

$$
\chi(S)_i = S_i \oplus \big(\neg S_{(i+1) \bmod b} \;\wedge\; S_{(i+2) \bmod b}\big)
$$

**Purpose**: introduces non-linearity, making the function resistant to linear and differential cryptanalysis.  
**In real Keccak**: identical formula, applied to 64-bit lanes over the 5×5 state matrix.  
**Quantum mapping**: each bit requires one **Toffoli gate** (controlled-controlled-NOT). This is the most expensive step in a quantum circuit — Toffoli gates dominate the oracle cost in a Grover attack.

---

#### $\iota$ — Round constant injection

XORs each bit of the state with a round constant $RC$:

$$
\iota(S, RC)_i = S_i \oplus RC_i
$$

**Purpose**: breaks symmetry between rounds, preventing slide attacks.  
**In real Keccak**: only lane $(0,0)$ is XORed with a 64-bit LFSR-derived constant — here we apply a fixed constant to the full state for simplicity.  
**Quantum mapping**: implementable with CNOT gates (one per bit where $RC_i = 1$).

---

The full permutation per round is:

$$
f_j(S) = \iota\big(RC_j,\; \chi(\rho(S))\big)
$$

 > The functions defined below are available as a standalone module in [`keccak_toy.py`](keccak_toy.py).  


In [5]:
import sys

In [6]:
sys.path.append('..')

In [7]:
from lib.keccak_toy import rho, chi, iota, keccak_toy

### 4️⃣ Toy Keccak Sponge Function

$$
\begin{aligned}
S &\gets 0^b && \text{Initialize state to all zeros of size $b$} \\
S &\gets S \oplus (M \, \| \, 0^c) && \text{Absorb the message $M$ padded with $c$ zeros} \\
\text{for } j &= 0, \dots, n_r-1: && \text{Apply $n_r$ rounds of permutation} \\
\quad S &\gets \iota \, RC_j \big( \chi(\rho(S)) \big) && \text{Apply the $\rho$, $\chi$, and round constant $\iota RC_j$ steps} \\
H(M) &= \mathrm{Trunc}_\ell(S) && \text{Output hash by truncating the state to length $\ell$}
\end{aligned}
$$

In [8]:
# Example
input_bits = [1,0,1,1]
hash_output = keccak_toy(input_bits)
print('Toy Keccak hash output (4 bits):', hash_output)

Toy Keccak hash output (4 bits): [0, 1, 1, 1]


### 5️⃣ Testing Multiple Inputs


In [9]:
inputs: list = [
    [0,0,0,0],
    [1,1,1,1],
    [1,0,1,0]
]

for inp in inputs:
    out = keccak_toy(inp)
    print(f'Input: {inp} -> Toy Keccak Hash: {out}')

Input: [0, 0, 0, 0] -> Toy Keccak Hash: [1, 0, 1, 0]
Input: [1, 1, 1, 1] -> Toy Keccak Hash: [0, 1, 1, 1]
Input: [1, 0, 1, 0] -> Toy Keccak Hash: [0, 0, 1, 1]


### 6️⃣ Step-by-Step State Visualization

In [10]:
state = [0]*B
print('Initial state:\n', state)
state = [s ^ i for s,i in zip(input_bits + [0]*(B-len(input_bits)), state)]
print('After absorption:\n', state)
for idx, RC in enumerate([[1,0,1,0,1,0,1,0]]*N_ROUNDS):
    state = rho(state)
    print(f'After rho round {idx+1}:\n', state)
    state = chi(state)
    print(f'After chi round {idx+1}:\n', state)
    state = iota(state, RC)
    print(f'After iota round {idx+1}:\n', state)
hash_output = state[:L]
print('Final output (squeeze):\n', hash_output)

Initial state:
 [0, 0, 0, 0, 0, 0, 0, 0]
After absorption:
 [1, 0, 1, 1, 0, 0, 0, 0]
After rho round 1:
 [0, 1, 1, 0, 0, 0, 0, 1]
After chi round 1:
 [0, 1, 1, 0, 0, 1, 0, 0]
After iota round 1:
 [1, 1, 0, 0, 1, 1, 1, 0]
After rho round 2:
 [1, 0, 0, 1, 1, 1, 0, 1]
After chi round 2:
 [1, 1, 0, 1, 1, 0, 0, 1]
After iota round 2:
 [0, 1, 1, 1, 0, 0, 1, 1]
Final output (squeeze):
 [0, 1, 1, 1]


### 7️⃣ Notes on Formal Properties

- The toy function $f$ is a permutation of $\{0,1\}^b$ (invertible).  
- The sponge construction: absorption ($\oplus$), followed by permutation, then squeeze.  
- The toy hash length $\ell = 4$ is not secure, but sufficient for educational purposes and small Grover simulations.  
- Each operation can be mapped to reversible quantum gates (CNOT, Toffoli, SWAP) for Grover oracle construction.

In [11]:
from ipywidgets import interact, ToggleButtons
from typing import Literal

In [12]:
bit = Literal[0, 1]

In [13]:
def interactive_keccak(bit0: bit, bit1: bit, bit2: bit, bit3: bit) -> None:
    def bits_to_str(bits):
        return ''.join(str(b) for b in bits)
    
    input_bits = [bit0, bit1, bit2, bit3]
    state = [0]*8
    
    print("="*40)
    print("Keccak Toy Step-by-Step Visualization")
    print("="*40, "\n")
    
    print(f"Input bits:        {bits_to_str(input_bits)}")
    print(f"Initial state:     {bits_to_str(state)}\n")
    
    # Absorption
    state = [s ^ i for s,i in zip(input_bits + [0]*4, state)]
    print(f"After absorption:  {bits_to_str(state)}\n")
    
    # Rounds
    for idx, RC in enumerate([[1,0,1,0,1,0,1,0]]*2):
        print(f"--- Round {idx+1} ---")
        state = rho(state)
        print(f"ρ (rho):         {bits_to_str(state)}")
        state = chi(state)
        print(f"χ (chi):         {bits_to_str(state)}")
        state = iota(state, RC)
        print(f"ι (iota):        {bits_to_str(state)}\n")
    
    output_bits = state[:4]
    print(f"Final output (squeeze, {len(output_bits)} bits): {bits_to_str(output_bits)}")
    print("="*40)

In [14]:
# Widget interactivo
interact(
    interactive_keccak,
    bit0=ToggleButtons(options=[0, 1], description='b0'),
    bit1=ToggleButtons(options=[0, 1], description='b1'),
    bit2=ToggleButtons(options=[0, 1], description='b2'),
    bit3=ToggleButtons(options=[0, 1], description='b3')
)

interactive(children=(ToggleButtons(description='b0', options=(0, 1), value=0), ToggleButtons(description='b1'…

<function __main__.interactive_keccak(bit0: Literal[0, 1], bit1: Literal[0, 1], bit2: Literal[0, 1], bit3: Literal[0, 1]) -> None>

### 8️⃣ Real Keccak-256

<style>
    .python-function {
        color: #228B22;
        font-family: monospace;
    }
</style>

Full 1600-bit state, 24 rounds, 5-step permutation (θ, ρ, π, χ, ι).</br>
This is the algorithm Ethereum uses for address derivation and 
<span class="python-function">keccak()</span> in Solidity.</br>  
Note: SHA3-256 differs only in the padding byte <span class="python-function">0x06</span> vs <span class="python-function">0x01</span>.

In [15]:
# 24 round constants (generated by LFSR)
KECCAK_RC = [
    0x0000000000000001, 0x0000000000008082, 0x800000000000808A, 0x8000000080008000,
    0x000000000000808B, 0x0000000080000001, 0x8000000080008081, 0x8000000000008009,
    0x000000000000008A, 0x0000000000000088, 0x0000000080008009, 0x000000008000000A,
    0x000000008000808B, 0x800000000000008B, 0x8000000000008089, 0x8000000000008003,
    0x8000000000008002, 0x8000000000000080, 0x000000000000800A, 0x800000008000000A,
    0x8000000080008081, 0x8000000000008080, 0x0000000080000001, 0x8000000080008008,
]

# ρ rotation offsets for each of the 25 lanes (5x5)
KECCAK_RHO = [
    [ 0, 36,  3, 41, 18],
    [ 1, 44, 10, 45,  2],
    [62,  6, 43, 15, 61],
    [28, 55, 25, 21, 56],
    [27, 20, 39,  8, 14],
]

#### Round Constants & Rotation Offsets

**`KECCAK_RC`** — 24 round constants derived from a binary LFSR. Each is a 64-bit value XORed into lane $(0,0)$ during $\iota$ to break symmetry between rounds.

**`KECCAK_RHO`** — 5×5 matrix of rotation offsets for $\rho$. Each lane $(x, y)$ in the state is rotated left by a different fixed amount, providing diffusion across the 64-bit lane width.

#### The 5-Step Permutation `keccak_f[1600]`

Each of the 24 rounds applies 5 transformations in order:

| Step | Symbol | Description |
|------|--------|-------------|
| Theta | $\theta$ | XOR each lane with the parity of two adjacent columns — linear diffusion |
| Rho | $\rho$ | Rotate each lane left by a fixed offset — intra-lane diffusion |
| Pi | $\pi$ | Permute lane positions in the 5×5 matrix — inter-lane diffusion |
| Chi | $\chi$ | $A_i \oplus (\neg A_{i+1} \wedge A_{i+2})$ — the **only non-linear** step, identical in structure to the toy |
| Iota | $\iota$ | XOR lane $(0,0)$ with the round constant — breaks round symmetry |

$$
\text{Round}_j(A) = \iota_{RC_j}(\chi(\pi(\rho(\theta(A)))))
$$

In [16]:
def rot64(x: int, n: int) -> int:
    return ((x << n) | (x >> (64 - n))) & 0xFFFFFFFFFFFFFFFF

def keccak_f1600(state: list) -> list:
    """24 rounds of Keccak-f[1600] on a 5×5 matrix of 64-bit lanes."""
    for rc in KECCAK_RC:
        # θ (theta): XOR each lane with parity of two adjacent columns
        C = [state[x][0] ^ state[x][1] ^ state[x][2] ^ state[x][3] ^ state[x][4] for x in range(5)]
        D = [C[(x-1)%5] ^ rot64(C[(x+1)%5], 1) for x in range(5)]
        state = [[state[x][y] ^ D[x] for y in range(5)] for x in range(5)]

        # ρ (rho) + π (pi): rotate and permute lane positions
        B = [[0]*5 for _ in range(5)]
        for x in range(5):
            for y in range(5):
                B[y][(2*x + 3*y) % 5] = rot64(state[x][y], KECCAK_RHO[x][y])

        # χ (chi): non-linear mixing (same structure as toy — this is the key step)
        state = [[B[x][y] ^ ((~B[(x+1)%5][y]) & B[(x+2)%5][y]) for y in range(5)] for x in range(5)]

        # ι (iota): XOR lane (0,0) with round constant
        state[0][0] ^= rc

    return state

#### Sponge Construction

$$
\begin{aligned}
\text{Absorb:} \quad S &\gets \bigoplus_{i} \big( M_i \,\|\, 0^c \big) \xrightarrow{f_{1600}} \cdots \\
\text{Squeeze:} \quad H &= \mathrm{Trunc}_{256}(S)
\end{aligned}
$$

- **Rate** $r = 1088$ bits (136 bytes): the portion of state that gets XORed with message blocks  
- **Capacity** $c = 512$ bits: the protected portion that is never directly exposed  
- **Padding**: Keccak uses `0x01 ... 0x80` (SHA3 uses `0x06 ... 0x80` — this is why `keccak256` ≠ `sha3_256`)

In [17]:
def keccak256(message: bytes) -> str:
    """
    Keccak-256 hash. Returns lowercase hex string.
    Parameters: b=1600, r=1088, c=512, output=256 bits, 24 rounds.
    """
    rate_bytes = 136  # r = 1088 bits / 8

    # 10*1 padding (Keccak standard — NOT SHA3 which uses 0x06)
    msg = bytearray(message)
    msg.append(0x01)
    while len(msg) % rate_bytes != 0:
        msg.append(0x00)
    msg[-1] |= 0x80

    # Initialize 5×5 state
    state = [[0]*5 for _ in range(5)]

    # Absorb phase: XOR message blocks into state, apply permutation
    for block_start in range(0, len(msg), rate_bytes):
        block = msg[block_start:block_start + rate_bytes]
        for i in range(rate_bytes // 8):
            x, y = i % 5, i // 5
            lane = int.from_bytes(block[i*8:(i+1)*8], 'little')
            state[x][y] ^= lane
        state = keccak_f1600(state)

    # Squeeze phase: extract first 256 bits (4 lanes × 8 bytes)
    output = bytearray()
    for i in range(4):
        output += state[i % 5][i // 5].to_bytes(8, 'little')

    return output.hex()

In [18]:
# Known test vectors
assert keccak256(b'') == 'c5d2460186f7233c927e7db2dcc703c0e500b653ca82273b7bfad8045d85a470', "Empty string vector failed"
assert keccak256(b'abc') == '4e03657aea45a94fc7d47ba826c8d667c0d1e6e33a64a036ec44f58fa12d6c45', "abc vector failed"

print("✓ All test vectors pass\n")
print(f"keccak256('')    = {keccak256(b'')}")
print(f"keccak256('abc') = {keccak256(b'abc')}")
print(f"keccak256('hello ethereum') = {keccak256(b'hello ethereum')}")

✓ All test vectors pass

keccak256('')    = c5d2460186f7233c927e7db2dcc703c0e500b653ca82273b7bfad8045d85a470
keccak256('abc') = 4e03657aea45a94fc7d47ba826c8d667c0d1e6e33a64a036ec44f58fa12d6c45
keccak256('hello ethereum') = b466f598d977fdbe7eea49ac6be9080b8f74f450c2f7ad3d42b98c54f9d07cc9


#### Toy vs Real Keccak-256

| | Toy (this notebook) | Real Keccak-256 |
|---|---|---|
| State size | 8 bits (1D list) | 1600 bits (5×5×64 lanes) |
| Rounds | 2 | 24 |
| $\theta$ (theta) | ❌ | ✓ column parity diffusion |
| $\rho$ (rho) | shift by 1 | rotate by lane-specific offset |
| $\pi$ (pi) | ❌ | ✓ position permutation |
| $\chi$ (chi) | ✓ same formula | ✓ same formula (64-bit lanes) |
| $\iota$ (iota) | fixed constant | 24 unique LFSR-derived constants |
| Output | 4 bits | 256 bits |
| Use | Grover simulation | Ethereum `keccak256()`, address derivation |

</br>

> The `chi` step is structurally **identical** in both — it is the core of quantum hardness analysis.